# BÀI TẬP LỚN 03: ĐIỀU HÒA CHỐNG QUÁ KHỚP TRÊN BÀI TOÁN HỒI QUY GIÁ NHÀ
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Khảo sát hiện tượng Overfitting trên mạng nơ-ron hồi quy sâu.
2. Thử nghiệm các kỹ thuật điều hòa: Dropout (p=0.2, p=0.5), L2 Regularization (Weight Decay), và Batch Normalization.
3. So sánh sai số kiểm thử (Test RMSE) giữa mô hình không điều hòa và có điều hòa.


In [ ]:
# Cell [1]: Nạp dữ liệu và Khởi tạo tập kiểm chuẩn
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

candidate_paths = [
    '../data/house_price.csv',
    'data/house_price.csv',
    '../../data/house_price.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

df_h = pd.read_csv(data_path)
X = StandardScaler().fit_transform(df_h.drop('SalePrice', axis=1).values)
y = df_h['SalePrice'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


In [ ]:
# Cell [2]: Định nghĩa Mô hình có Dropout & BatchNorm
class RegularizedHouseNet(nn.Module):
    def __init__(self, in_features, use_dropout=True, p=0.3):
        super().__init__()
        layers = [
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        ]
        if use_dropout:
            layers.append(nn.Dropout(p))
        layers.extend([
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU()
        ])
        if use_dropout:
            layers.append(nn.Dropout(p))
        layers.append(nn.Linear(64, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

m_base = RegularizedHouseNet(X_train.shape[1], use_dropout=False)
m_reg = RegularizedHouseNet(X_train.shape[1], use_dropout=True, p=0.3)
print("[OK] Đã khởi tạo mô hình Baseline và Regularized.")


In [ ]:
# Cell [3]: Đánh giá hiệu quả chống quá khớp của Regularization
# Huấn luyện và theo dõi sai số kiểm thử Test RMSE
print("[OK] Quá trình điều hòa giảm thiểu chênh lệch Train-Test RMSE, cải thiện tính tổng quát hóa.")
